# 🏥 Healthcare Payment Variance Analytics & Contract Modeling Simulator
### Project 1 — Guidehouse Healthcare Analytics | May 1, 2026
**Author:** Peter Chika Ozo-ogueji | American University | po3783a@american.edu

---
## 📋 Table of Contents
1. [Setup & Installation](#setup)
2. [Mount Google Drive & Configuration](#drive)
3. [Load All Datasets](#load)
4. [Data Validation](#validate)
5. [MySQL Connection & Raw Layer](#mysql)
6. [DuckDB Analytical Engine](#duckdb)
7. [IPPS Expected Payment Formula](#ipps)
8. [Payment Variance Computation](#variance)
9. [SQL Analytics — Window Functions & CTEs](#sql)
10. [OPPS Outpatient Layer](#opps)
11. [NY SPARCS Multi-Payer Analysis](#sparcs)
12. [Contract Scenario Modeling](#scenarios)
13. [Visualizations](#viz)
14. [Export (Excel, Parquet, MySQL)](#export)
15. [Executive Summary](#summary)

> **Critical rate correction:** FY2025 IPPS Base Rate = **$6,624.39** (IFC Oct 2024), NOT $6,394.36
> IME + DSH = **additive** `(1 + IME + DSH)` — never multiplicative


## 1. Setup & Installation <a id='setup'></a>

In [ ]:
# Install all required packages
import subprocess, sys
pkgs = ["duckdb>=1.1.0","pandera>=0.20.0","sqlalchemy>=2.0","pymysql",
        "pyarrow>=15.0","plotly>=5.20","openpyxl","tqdm","PyYAML"]
for p in pkgs:
    subprocess.check_call([sys.executable,"-m","pip","install",p,"-q"])
print("✅ All packages installed")


In [ ]:
import os, sys, logging, warnings, re, urllib.parse
from pathlib import Path
import numpy as np
import pandas as pd
import duckdb
import pandera as pa
from pandera import Column, DataFrameSchema, Check
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

pio.renderers.default = "colab"
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.2f}".format)

logging.basicConfig(level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)])
log = logging.getLogger("pva")
log.info("✅ Imports complete | pandas %s | duckdb %s", pd.__version__, duckdb.__version__)


## 2. Mount Google Drive & Configuration <a id='drive'></a>

In [ ]:
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive', force_remount=False)
log.info("✅ Google Drive mounted")


In [ ]:
# ⚠️  UPDATE THIS PATH to match your Google Drive folder name exactly
GDRIVE_BASE = Path("/content/drive/MyDrive/Guidehouse-Data-Science")

PATHS = {
    "inp_provider":   GDRIVE_BASE / "CMS Inpatient Provider FY2022" / "CMS_Inpatient_Provider_FY2022_2024.csv",
    "out_provider":   GDRIVE_BASE / "CMS Outpatient Provider CY2022" / "MUP_OUT_RY25_P04_V10_DY23_Prov_Svc.csv",
    "pfs_rvu":        GDRIVE_BASE / "CY2025 PFS RVU25A" / "PPRRVU25_JAN.csv",
    "ipps_t5_cn":     GDRIVE_BASE / "IPPS FY2025 Table 5" / "IPPS_FY2025_Table_5_CN.csv",
    "ipps_t5_fr":     GDRIVE_BASE / "IPPS FY2025 Table 5" / "IPPS_FY2025_Table_5_FR.csv",
    "ipps_impact":    GDRIVE_BASE / "IPPS_Impact_File_main" / "FY_2025_IPPS_FR_FY_2025_Final.csv",
    "ipps_impact_cn": GDRIVE_BASE / "IPPS_Impact_File_main" / "FY_2025_IPPS_FY_2025_CN.csv",
    "ipps_impact_ifc":GDRIVE_BASE / "IPPS_Impact_File_main" / "FY_2025_IPPS_FY_2025_IFC.csv",
    "sparcs":         GDRIVE_BASE / "NY_SPARCS_2022_Inpatient" / "Hospital_Inpatient_Discharges_(SPARCS_De-Identified)__2022_20260427.csv",
    "opps_addenda_p": GDRIVE_BASE / "OPPS CY2025 AddendaMain" / "OPPS_CY2025_Addenda_P.csv",
    "opps_addenda_q": GDRIVE_BASE / "OPPS CY2025 AddendaMain" / "OPPS_CY2025_Addenda_Q.csv",
}
OUTPUT_DIR = GDRIVE_BASE / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("📁 File verification:")
for name, path in PATHS.items():
    status = "✅" if path.exists() else "❌ MISSING"
    print(f"  {status}  {name}: {path.name}")


In [ ]:
# ── FY2025 IPPS CONSTANTS (IFC-corrected, 89 FR 80452, Oct 3 2024) ────────────
# Source: CMS-1808-IFC post Bridgeport Hospital v. Becerra D.C. Circuit ruling
IPPS = {
    "BASE_RATE":            6_624.39,   # ← CORRECT (not $6,394.36)
    "LABOR_SHARE_HIGH_WI":  0.676,      # Wage index > 1.0
    "LABOR_SHARE_LOW_WI":   0.620,      # Wage index <= 1.0 (statutory)
    "CAPITAL_RATE":         512.14,
    "OUTLIER_THRESHOLD":    46_217.0,
    "OUTLIER_MCF_STD":      0.80,
    "SEQUESTRATION":        0.98,
}
PFS_CF           = 32.35      # CY2025 physician fee schedule conversion factor
YIELD_THRESHOLD  = 0.92       # Claims below 92% yield = underpaid
STOP_LOSS_THRESH = 85_000.0   # Stop-loss threshold for outlier flagging

print(f"✅ IPPS Base Rate: ${IPPS['BASE_RATE']:,.2f} | PFS CF: ${PFS_CF:.2f} | Outlier: ${IPPS['OUTLIER_THRESHOLD']:,.2f}")


## 3. Load All Datasets <a id='load'></a>

In [ ]:
# ── Reusable helpers ──────────────────────────────────────────────────────────
def parse_money(s: pd.Series) -> pd.Series:
    """Parse currency strings: $, commas, parenthesised negatives."""
    return pd.to_numeric(
        s.astype(str).str.strip()
         .str.replace(r'[$,]', '', regex=True)
         .str.replace(r'^\((.+)\)$', r'-\1', regex=True)
         .str.replace(' ', '', regex=False),
        errors='coerce')

def norm_ccn(s: pd.Series) -> pd.Series:
    """6-char zero-padded string CCN — NEVER cast to int."""
    return s.astype(str).str.strip().str.zfill(6)

def norm_drg(s: pd.Series, width: int = 3) -> pd.Series:
    """Zero-padded DRG/APC code — NEVER cast to int."""
    return s.astype(str).str.strip().str.zfill(width)

def load_csv(path: Path, label: str, skiprows: int = 0) -> pd.DataFrame:
    """Load CMS CSV with UTF-8-BOM → CP1252 → Latin-1 fallback."""
    for enc in ['utf-8-sig','utf-8','cp1252','latin-1']:
        try:
            df = pd.read_csv(path, dtype=str, skiprows=skiprows,
                             encoding=enc, low_memory=False,
                             engine='pyarrow', on_bad_lines='warn')
            df.columns = df.columns.str.strip()
            df['_source']   = label
            df['_dq_flags'] = [set() for _ in range(len(df))]
            log.info("✅ %-26s | %7d rows × %d cols | %s", label, len(df), df.shape[1], enc)
            return df
        except Exception:
            continue
    raise RuntimeError(f"Cannot load {path}")

print("✅ Helper functions ready")


In [ ]:
# ── 3a. CMS Inpatient Provider FY2022 ────────────────────────────────────────
df_inp = load_csv(PATHS["inp_provider"], "CMS_INP_FY2022")
df_inp['Rndrng_Prvdr_CCN'] = norm_ccn(df_inp['Rndrng_Prvdr_CCN'])
df_inp['DRG_Cd']           = norm_drg(df_inp['DRG_Cd'], 3)
for col in ['Avg_Submtd_Cvrd_Chrg','Avg_Tot_Pymt_Amt','Avg_Mdcr_Pymt_Amt']:
    if col in df_inp.columns:
        df_inp[col] = parse_money(df_inp[col])
df_inp['Tot_Dschrgs'] = pd.to_numeric(df_inp['Tot_Dschrgs'], errors='coerce')
print(df_inp[['Rndrng_Prvdr_CCN','DRG_Cd','Tot_Dschrgs','Avg_Submtd_Cvrd_Chrg','Avg_Mdcr_Pymt_Amt']].head(3))


In [ ]:
# ── 3b. IPPS FY2025 Table 5 (CN file supersedes FR) ─────────────────────────
def load_ipps_table5(path, label):
    """Skip metadata rows; find header row containing MS-DRG."""
    raw = pd.read_csv(path, header=None, dtype=str,
                      encoding='utf-8-sig', engine='python', on_bad_lines='skip')
    hrow = next((i for i, r in raw.iterrows()
                 if any('MS-DRG' in str(v).upper() or 'MSDRG' in str(v).upper().replace('-','')
                        for v in r.values)), None)
    if hrow is None:
        raise ValueError(f"No MS-DRG header in {path}")
    df = pd.read_csv(path, skiprows=hrow, dtype=str,
                     encoding='utf-8-sig', engine='python', on_bad_lines='skip')
    df.columns = df.columns.str.strip()
    rename = {}
    for col in df.columns:
        c = col.upper().replace(' ','_').replace('-','_')
        if 'MS' in c and 'DRG' in c and 'DRG_CD' not in rename.values(): rename[col]='DRG_CD'
        elif c=='MDC':                                                      rename[col]='MDC'
        elif 'TYPE' in c and len(rename.get(col,''))<5:                    rename[col]='TYPE'
        elif 'WEIGHT' in c or c=='WEIGHTS':                                rename[col]='WEIGHT'
        elif 'GEOMETRIC' in c or 'GMLOS' in c:                             rename[col]='GMLOS'
        elif 'ARITHMETIC' in c or 'AMLOS' in c:                            rename[col]='AMLOS'
        elif 'TITLE' in c or 'DESC' in c:                                  rename[col]='DRG_DESC'
    df = df.rename(columns=rename)
    if 'DRG_CD' in df.columns: df['DRG_CD'] = norm_drg(df['DRG_CD'], 3)
    for col in ['WEIGHT','GMLOS','AMLOS']:
        if col in df.columns: df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.dropna(subset=['WEIGHT']).reset_index(drop=True)
    df['_source']   = label
    df['_dq_flags'] = [set() for _ in range(len(df))]
    log.info("✅ %-26s | %7d DRGs | weight range: %.3f – %.3f",
             label, len(df), df['WEIGHT'].min(), df['WEIGHT'].max())
    return df

try:
    df_t5 = load_ipps_table5(PATHS["ipps_t5_cn"], "IPPS_T5_CN")
except Exception:
    df_t5 = load_ipps_table5(PATHS["ipps_t5_fr"], "IPPS_T5_FR")
    log.warning("CN failed — using FR weights")

print(df_t5[['DRG_CD','DRG_DESC','WEIGHT','GMLOS','MDC','TYPE']].head(5))


In [ ]:
# ── 3c. IPPS Impact File (IFC > CN > Final priority) ────────────────────────
def load_impact(path, label):
    df = load_csv(path, label)
    df.columns = df.columns.str.strip().str.upper().str.replace(' ','_')
    for p in ['PROVIDER','CCN','PRVDR_NUM']:
        if p in df.columns: df['CCN'] = norm_ccn(df[p]); break
    for std,candidates in {
        'WAGE_INDEX':['WI','WAGE_INDEX','CBSA_WI'],
        'IME_ADJ':   ['IME_ADJ','IME_ADJ_FACTOR','IME_FACTOR'],
        'DSH_ADJ':   ['DSH_ADJ','DSH_OPER_ADJ','DSHPCT'],
        'BEDS':      ['BEDS','NUM_BEDS']}.items():
        for c in candidates:
            if c in df.columns: df[std]=pd.to_numeric(df[c],errors='coerce'); break
    return df

for key in ['ipps_impact_ifc','ipps_impact_cn','ipps_impact']:
    try:
        df_impact = load_impact(PATHS[key], f"IMPACT_{key.upper()}")
        log.info("Impact file: %s", key)
        break
    except Exception as e:
        log.warning("Skip %s: %s", key, e)

df_impact['WAGE_INDEX'] = df_impact['WAGE_INDEX'].fillna(1.0)
df_impact['IME_ADJ']    = df_impact['IME_ADJ'].fillna(0.0)
df_impact['DSH_ADJ']    = df_impact['DSH_ADJ'].fillna(0.0)
print(df_impact[['CCN','WAGE_INDEX','IME_ADJ','DSH_ADJ']].dropna().head(5))


In [ ]:
# ── 3d. PFS RVU25A ────────────────────────────────────────────────────────────
df_pfs = load_csv(PATHS["pfs_rvu"], "PFS_RVU25A")
df_pfs.columns = df_pfs.columns.str.strip().str.upper().str.replace(' ','_')
for c in ['TOTAL_RVU','TOTAL RVU','TRVU','TOT_RVU']:
    if c in df_pfs.columns:
        df_pfs['TOTAL_RVU_STD'] = pd.to_numeric(df_pfs[c], errors='coerce')
        df_pfs['MEDICARE_PMT']  = df_pfs['TOTAL_RVU_STD'] * PFS_CF
        break
print(f"PFS: {len(df_pfs):,} rows | CF=${PFS_CF}")
print(df_pfs.head(3))

# ── 3e. OPPS Addenda ──────────────────────────────────────────────────────────
df_opps_p = load_csv(PATHS["opps_addenda_p"], "OPPS_ADDENDA_P")
df_opps_q = load_csv(PATHS["opps_addenda_q"], "OPPS_ADDENDA_Q")
print(f"\nOPPS Addenda P: {len(df_opps_p):,} rows | Q: {len(df_opps_q):,} rows")

# ── 3f. CMS Outpatient Provider ───────────────────────────────────────────────
df_out = load_csv(PATHS["out_provider"], "CMS_OUT_CY2022")
for c in ['Rndrng_Prvdr_CCN','CCN']:
    if c in df_out.columns: df_out['Rndrng_Prvdr_CCN']=norm_ccn(df_out[c]); break
for col in [c for c in df_out.columns if any(x in c.lower() for x in ['pymt','alowd','allowed'])]:
    df_out[col] = parse_money(df_out[col])
print(f"Outpatient: {len(df_out):,} rows")


In [ ]:
# ── 3g. NY SPARCS 2022 (2.4M rows — allow ~60s) ──────────────────────────────
log.info("Loading NY SPARCS 2022...")
df_sparcs = load_csv(PATHS["sparcs"], "NY_SPARCS_2022")

rename_sparcs = {}
for col in df_sparcs.columns:
    c = col.strip()
    if 'APR DRG Code' in c:         rename_sparcs[col]='APR_DRG_CD'
    elif 'APR MDC Code' in c:       rename_sparcs[col]='APR_MDC_CD'
    elif 'APR MDC Description' in c:rename_sparcs[col]='MDC_DESC'
    elif 'APR Severity' in c:       rename_sparcs[col]='SEVERITY'
    elif 'Payment Typology 1' in c: rename_sparcs[col]='PAY_TYPE_1'
    elif 'Payment Typology 2' in c: rename_sparcs[col]='PAY_TYPE_2'
    elif 'Payment Typology 3' in c: rename_sparcs[col]='PAY_TYPE_3'
    elif 'Total Charges' in c:      rename_sparcs[col]='TOTAL_CHARGES'
    elif 'Total Costs' in c:        rename_sparcs[col]='TOTAL_COSTS'
    elif 'Length of Stay' in c:     rename_sparcs[col]='LOS'
    elif 'Permanent Facility' in c: rename_sparcs[col]='FACILITY_ID'
    elif 'Facility Name' in c:      rename_sparcs[col]='FACILITY_NAME'
    elif 'APR DRG Description' in c:rename_sparcs[col]='APR_DRG_DESC'

df_sparcs = df_sparcs.rename(columns=rename_sparcs)
for col in ['TOTAL_CHARGES','TOTAL_COSTS']:
    if col in df_sparcs.columns: df_sparcs[col]=parse_money(df_sparcs[col])
if 'LOS' in df_sparcs.columns:
    df_sparcs['LOS'] = pd.to_numeric(
        df_sparcs['LOS'].astype(str).str.replace('+','',regex=False), errors='coerce')
if 'APR_DRG_CD' in df_sparcs.columns:
    df_sparcs['APR_DRG_CD'] = norm_drg(df_sparcs['APR_DRG_CD'], 3)

PAYER_MAP = {
    'Medicare':'Medicare','Medicaid':'Medicaid',
    'Blue Cross/Blue Shield':'Commercial_BCBS',
    'Private Health Insurance':'Commercial_Private',
    'Managed Care, Unspecified':'Commercial_MC',
    'Self-Pay':'Self_Pay','Federal/State/Local/VA':'Federal',
    'Department of Corrections':'Corrections',
    'Miscellaneous/Other':'Other','Unknown':'Unknown'}
if 'PAY_TYPE_1' in df_sparcs.columns:
    df_sparcs['PAYER_TIER'] = df_sparcs['PAY_TYPE_1'].map(PAYER_MAP).fillna('Other')

print(f"\nNY SPARCS 2022: {len(df_sparcs):,} discharges")
if 'PAYER_TIER' in df_sparcs.columns:
    print(df_sparcs['PAYER_TIER'].value_counts().head(8))


## 4. Data Validation (Pandera) <a id='validate'></a>

In [ ]:
# ── Pandera schema validation ────────────────────────────────────────────────
inp_schema = DataFrameSchema({
    "Rndrng_Prvdr_CCN":     Column(str,  nullable=False),
    "DRG_Cd":               Column(str,  nullable=False),
    "Tot_Dschrgs":          Column(float,Check.ge(0), nullable=True),
    "Avg_Submtd_Cvrd_Chrg": Column(float,Check.ge(0), nullable=True),
    "Avg_Mdcr_Pymt_Amt":    Column(float,Check.ge(0), nullable=True),
})
try:
    inp_schema.validate(df_inp, lazy=True)
    log.info("✅ Inpatient schema validation PASSED")
except pa.errors.SchemaErrors as e:
    log.warning("⚠️  Schema errors: %d failures (non-fatal)", len(e.failure_cases))

def dq_report(df, label):
    print(f"\n{'='*55}\nDQ: {label} | {len(df):,} rows × {df.shape[1]} cols")
    null_counts = df.select_dtypes(include='number').isnull().sum()
    for col, n in null_counts[null_counts > 0].items():
        print(f"  NULL  {col}: {n:,} ({100*n/len(df):.1f}%)")
    for col in df.select_dtypes(include='number').columns:
        if any(k in col.lower() for k in ['charg','pymt','amt','cost']):
            neg = (df[col]<0).sum()
            if neg > 0: print(f"  NEG   {col}: {neg:,}")

dq_report(df_inp,    "CMS Inpatient FY2022")
dq_report(df_t5,     "IPPS Table 5 FY2025")
dq_report(df_impact, "IPPS Impact File FY2025")


## 5. MySQL Connection & Raw Layer <a id='mysql'></a>

In [ ]:
# ── MySQL via Colab Secrets (set keys in the 🔑 icon in left sidebar) ────────
# Required secrets: MYSQL_HOST, MYSQL_USER, MYSQL_PWD, MYSQL_DB
from sqlalchemy import create_engine, text
from typing import Optional
from sqlalchemy.engine import Engine

def get_mysql_engine() -> Optional[Engine]:
    try:
        from google.colab import userdata
        host = userdata.get('MYSQL_HOST') or 'localhost'
        user = userdata.get('MYSQL_USER') or 'root'
        pwd  = urllib.parse.quote_plus(userdata.get('MYSQL_PWD') or '')
        db   = userdata.get('MYSQL_DB')  or 'healthcare_pva'
    except Exception:
        import getpass
        host = input("MySQL host: ") or 'localhost'
        user = input("MySQL user: ") or 'root'
        pwd  = urllib.parse.quote_plus(getpass.getpass("MySQL password: "))
        db   = input("Database:   ") or 'healthcare_pva'
    url = f"mysql+pymysql://{user}:{pwd}@{host}/{db}?charset=utf8mb4"
    try:
        engine = create_engine(url, pool_pre_ping=True, pool_recycle=3600)
        with engine.connect() as c: c.execute(text("SELECT 1"))
        log.info("✅ MySQL connected → %s@%s/%s", user, host, db)
        return engine
    except Exception as e:
        log.warning("⚠️  MySQL unavailable: %s", e)
        return None

engine = get_mysql_engine()
MYSQL_OK = engine is not None
print(f"MySQL: {'connected ✅' if MYSQL_OK else 'not connected ❌ (continuing without)'}")


In [ ]:
def push_mysql(df: pd.DataFrame, table: str, engine: Engine, chunksize=2000) -> None:
    if not engine: log.warning("MySQL not available — skip %s", table); return
    dfc = df.drop(columns=['_dq_flags','_source'], errors='ignore').copy()
    for col in dfc.select_dtypes(include='object').columns:
        dfc[col] = dfc[col].astype(str).str[:255]
    dfc.to_sql(table, engine, if_exists='replace', index=False,
               chunksize=chunksize, method='multi')
    log.info("✅ MySQL ← %d rows → `%s`", len(dfc), table)

if MYSQL_OK:
    push_mysql(df_inp.head(10000), "raw_cms_inpatient",  engine)
    push_mysql(df_t5,              "raw_ipps_t5",        engine)
    push_mysql(df_impact,          "raw_ipps_impact",    engine)


## 6. DuckDB Analytical Engine <a id='duckdb'></a>

In [ ]:
con = duckdb.connect(database=':memory:')
con.register('inp_raw',    df_inp)
con.register('t5_raw',     df_t5)
con.register('impact_raw', df_impact)
con.register('out_raw',    df_out)
con.register('pfs_raw',    df_pfs)
con.register('opps_p',     df_opps_p)
con.register('sparcs_raw', df_sparcs)
tables = con.execute("SHOW TABLES").fetchdf()
print("✅ DuckDB tables registered:\n", tables)


## 7. IPPS Expected Payment Formula <a id='ipps'></a>

### Formula (FY2025 IFC-corrected):
```
Operating Payment =
    [(Labor_Share × WageIndex) + NonLabor_Share]
    × $6,624.39
    × DRG_Weight
    × (1 + IME_Adj + DSH_Adj)    ← ADDITIVE (not multiplicative!)
    × 0.98                        ← Sequestration
```


In [ ]:
# ── Build master join: Inpatient × Table5 × ImpactFile ───────────────────────
df_master = con.execute("""
    SELECT
        i.Rndrng_Prvdr_CCN                      AS ccn,
        i.DRG_Cd                                AS drg_cd,
        COALESCE(t5.DRG_DESC, i.DRG_Desc)      AS drg_desc,
        COALESCE(t5.MDC, 'XX')                  AS mdc,
        COALESCE(t5.TYPE,'UNK')                 AS drg_type,
        TRY_CAST(t5.WEIGHT AS DOUBLE)           AS drg_weight,
        TRY_CAST(t5.GMLOS  AS DOUBLE)           AS gmlos,
        i.Rndrng_Prvdr_State_Abrvtn             AS state,
        i.Rndrng_Prvdr_City                     AS city,
        TRY_CAST(i.Tot_Dschrgs           AS DOUBLE) AS tot_dschrgs,
        TRY_CAST(i.Avg_Submtd_Cvrd_Chrg  AS DOUBLE) AS avg_billed,
        TRY_CAST(i.Avg_Tot_Pymt_Amt      AS DOUBLE) AS avg_total_pymt,
        TRY_CAST(i.Avg_Mdcr_Pymt_Amt    AS DOUBLE) AS avg_medicare_pymt
    FROM inp_raw i
    LEFT JOIN t5_raw     t5  ON i.DRG_Cd     = t5.DRG_CD
""").df()

# Join impact file adjustments
df_adj = con.execute("""
    SELECT CCN AS ccn,
           TRY_CAST(WAGE_INDEX AS DOUBLE) AS wage_index,
           TRY_CAST(IME_ADJ    AS DOUBLE) AS ime_adj,
           TRY_CAST(DSH_ADJ    AS DOUBLE) AS dsh_adj,
           TRY_CAST(BEDS       AS DOUBLE) AS beds
    FROM impact_raw WHERE CCN IS NOT NULL
""").df()

df_master = df_master.merge(df_adj, on='ccn', how='left')
df_master['wage_index'] = df_master['wage_index'].fillna(1.0)
df_master['ime_adj']    = df_master['ime_adj'].fillna(0.0)
df_master['dsh_adj']    = df_master['dsh_adj'].fillna(0.0)
log.info("Master dataset: %d rows", len(df_master))
print(df_master[['ccn','drg_cd','drg_weight','wage_index','ime_adj','dsh_adj']].head(5))


In [ ]:
# ── Vectorised IPPS Expected Payment ─────────────────────────────────────────
def compute_ipps_expected(df: pd.DataFrame) -> pd.DataFrame:
    """
    FY2025 IPPS operating payment (IFC rates, additive IME+DSH).

    Operating = [(labor_share×WI) + non_labor] × BASE × weight × (1 + IME + DSH)
    Final     = Operating × Sequestration
    """
    df = df.copy()
    wi           = df['wage_index'].clip(lower=0.0)
    labor_share  = np.where(wi > 1.0, IPPS['LABOR_SHARE_HIGH_WI'], IPPS['LABOR_SHARE_LOW_WI'])
    nlabor_share = 1.0 - labor_share
    wage_adj     = (labor_share * wi + nlabor_share) * IPPS['BASE_RATE']
    ime_dsh      = 1.0 + df['ime_adj'].clip(lower=0.0) + df['dsh_adj'].clip(lower=0.0)

    df['expected_operating'] = wage_adj * df['drg_weight'] * ime_dsh
    df['expected_payment']   = df['expected_operating'] * IPPS['SEQUESTRATION']

    missing = df['drg_weight'].isna()
    df.loc[missing, 'expected_payment'] = np.nan
    return df

df_master = compute_ipps_expected(df_master)

print("✅ Expected payment statistics:")
print(df_master['expected_payment'].describe().apply("${:,.2f}".format))

# Verify DRG 470 (Major Hip/Knee - highest national volume)
for cd in ['470','0470']:
    sample = df_master[df_master['drg_cd']==cd]
    if len(sample):
        w = sample['drg_weight'].dropna().iloc[0]
        ep = sample['expected_payment'].dropna().iloc[0]
        print(f"\nDRG 470 check: weight={w:.4f} × $6,624.39 × seq → expected ~${ep:,.0f}")
        break


## 8. Payment Variance Computation <a id='variance'></a>

In [ ]:
def compute_variance(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute variance metrics with dual-threshold filtering and false-variance suppression.
    Dual threshold: flag if |Var%| > 5% AND |Var$| > $200 — prevents noise from dominating.
    """
    df = df.copy()
    df['expected_total'] = df['expected_payment'] * df['tot_dschrgs']
    df['actual_total']   = df['avg_medicare_pymt'] * df['tot_dschrgs']
    df['billed_total']   = df['avg_billed'] * df['tot_dschrgs']
    df['variance_dollars'] = df['expected_total'] - df['actual_total']
    df['variance_pct']     = df['variance_dollars'] / df['expected_total'].replace(0,np.nan) * 100
    df['yield_ratio']      = df['avg_medicare_pymt'] / df['expected_payment'].replace(0,np.nan)

    ABS_T, DOL_T = 5.0, 200.0
    df['underpayment_flag'] = ((df['variance_pct'] >  ABS_T) & (df['variance_dollars'] > DOL_T)).astype(int)
    df['overpayment_flag']  = ((df['variance_pct'] < -ABS_T) & (df['variance_dollars'] < -DOL_T)).astype(int)
    df['stop_loss_flag']    = (df['avg_billed'] > STOP_LOSS_THRESH).astype(int)
    df['high_priority']     = (df['yield_ratio'] < YIELD_THRESHOLD).astype(int)

    # Lesser-of suppression: if billed < expected AND actual ≥ billed×0.95 → compliant
    df['lesser_of_flag'] = ((df['avg_billed'] < df['expected_payment']) &
                             (df['avg_medicare_pymt'] >= df['avg_billed'] * 0.95)).astype(int)
    df.loc[df['lesser_of_flag']==1, 'underpayment_flag'] = 0

    und = df['underpayment_flag'].sum()
    ove = df['overpayment_flag'].sum()
    print(f"\n{'='*55}\nPAYMENT VARIANCE SUMMARY")
    print(f"  Total hospital×DRG pairs:    {len(df.dropna(subset=['variance_dollars'])):>10,}")
    print(f"  Underpayment flags:          {und:>10,}  ${df.loc[df['underpayment_flag']==1,'variance_dollars'].sum():>15,.0f}")
    print(f"  Overpayment flags:           {ove:>10,}  ${df.loc[df['overpayment_flag']==1,'variance_dollars'].abs().sum():>15,.0f}")
    print(f"  Lesser-of suppressed:        {df['lesser_of_flag'].sum():>10,}")
    print(f"  Stop-loss candidates:        {df['stop_loss_flag'].sum():>10,}")
    print(f"  High-priority (yield<92%):   {df['high_priority'].sum():>10,}")
    return df

df_variance = compute_variance(df_master)
con.register('variance', df_variance)
print("\n✅ Registered in DuckDB as 'variance'")


## 9. SQL Analytics — Window Functions & CTEs <a id='sql'></a>

In [ ]:
# ── Query 1: Top 20 DRGs by national underpayment ────────────────────────────
df_top_drgs = con.execute("""
    SELECT drg_cd, drg_desc, mdc, drg_type,
           ROUND(AVG(drg_weight),4)          AS avg_weight,
           SUM(tot_dschrgs)                  AS total_discharges,
           ROUND(SUM(variance_dollars),0)    AS total_underpayment,
           ROUND(AVG(yield_ratio)*100,2)     AS avg_yield_pct,
           COUNT(DISTINCT ccn)               AS num_hospitals
    FROM variance
    WHERE underpayment_flag=1 AND variance_dollars IS NOT NULL AND drg_weight IS NOT NULL
    GROUP BY drg_cd,drg_desc,mdc,drg_type
    ORDER BY total_underpayment DESC LIMIT 20
""").df()
print("🏆 TOP 20 DRGs BY UNDERPAYMENT")
print(df_top_drgs.to_string(index=False))


In [ ]:
# ── Query 2: State ranking — RANK() OVER window function ────────────────────
df_state_rank = con.execute("""
    WITH state_agg AS (
        SELECT state,
               ROUND(SUM(variance_dollars),0)   AS total_underpayment,
               ROUND(AVG(yield_ratio)*100,2)     AS avg_yield_pct,
               SUM(tot_dschrgs)                  AS total_discharges,
               COUNT(DISTINCT ccn)               AS hospitals
        FROM variance
        WHERE underpayment_flag=1 AND state NOT IN ('','nan','None')
        GROUP BY state
    )
    SELECT *, RANK() OVER (ORDER BY total_underpayment DESC) AS nat_rank
    FROM state_agg ORDER BY nat_rank LIMIT 30
""").df()
print("\n🗺️  STATE UNDERPAYMENT RANKING (WINDOW FUNCTION)")
print(df_state_rank.to_string(index=False))


In [ ]:
# ── Query 3: Payer yield CTE with priority tier ───────────────────────────────
df_yield = con.execute("""
    WITH drg_yield AS (
        SELECT drg_cd, drg_desc, mdc,
               ROUND(AVG(yield_ratio),4)         AS avg_yield,
               ROUND(STDDEV(yield_ratio),4)      AS std_yield,
               SUM(tot_dschrgs)                  AS vol,
               ROUND(SUM(variance_dollars),0)    AS total_var
        FROM variance
        WHERE yield_ratio IS NOT NULL AND drg_weight IS NOT NULL
        GROUP BY drg_cd,drg_desc,mdc HAVING vol >= 100
    )
    SELECT *, CASE
        WHEN avg_yield < 0.85 THEN 'CRITICAL (<85%)'
        WHEN avg_yield < 0.92 THEN 'HIGH (85-92%)'
        WHEN avg_yield < 0.96 THEN 'MONITOR (92-96%)'
        ELSE 'COMPLIANT (≥96%)' END AS priority_tier
    FROM drg_yield ORDER BY avg_yield ASC LIMIT 25
""").df()
print("\n📊 PAYER YIELD BENCHMARKING CTE — Bottom 25 DRGs")
print(df_yield.to_string(index=False))


In [ ]:
# ── Query 4: Case-Mix Index by state ─────────────────────────────────────────
df_cmi = con.execute("""
    SELECT state,
           ROUND(SUM(drg_weight*tot_dschrgs)/NULLIF(SUM(tot_dschrgs),0),4) AS cmi,
           SUM(tot_dschrgs)                AS total_discharges,
           ROUND(AVG(yield_ratio)*100,2)  AS avg_yield_pct
    FROM variance
    WHERE drg_weight IS NOT NULL AND state NOT IN ('','nan','None')
    GROUP BY state ORDER BY cmi DESC
""").df()
national_cmi = (df_variance['drg_weight']*df_variance['tot_dschrgs']).sum() / df_variance['tot_dschrgs'].sum()
print(f"\n📋 CASE-MIX INDEX BY STATE | National CMI: {national_cmi:.4f}")
print(df_cmi.to_string(index=False))

# ── Query 5: Stop-loss compliance & charge audit detection ───────────────────
df_sl = con.execute("""
    SELECT drg_cd, drg_desc, state, ccn,
           ROUND(avg_billed,0)           AS avg_billed,
           ROUND(expected_payment,0)     AS expected,
           ROUND(avg_medicare_pymt,0)    AS actual,
           ROUND(avg_billed-85000,0) AS over_threshold,
           ROUND(variance_dollars,0)     AS variance,
           ROUND(yield_ratio*100,2)      AS yield_pct,
           ROUND(avg_billed/85000*100,1) AS pct_of_threshold
    FROM variance WHERE stop_loss_flag=1 AND drg_weight IS NOT NULL
    ORDER BY over_threshold DESC LIMIT 20
""").df()
print("\n⚠️  STOP-LOSS OUTLIER ACCOUNTS (Top 20)")
print(df_sl.to_string(index=False))


## 10. OPPS Outpatient Layer <a id='opps'></a>

In [ ]:
# ── OPPS variance analysis ────────────────────────────────────────────────────
df_out_w = df_out.copy()
df_out_w.columns = df_out_w.columns.str.strip()

# Find payment and charge columns dynamically
pymt_col  = next((c for c in df_out_w.columns if any(x in c.lower() for x in ['pymt','payment','alowd'])), None)
chrg_col  = next((c for c in df_out_w.columns if any(x in c.lower() for x in ['sbmtd','chrg','charge'])), None)
svc_col   = next((c for c in df_out_w.columns if any(x in c.lower() for x in ['srvcs','capc','count'])), None)

log.info("Outpatient cols — payment: %s | charges: %s | services: %s", pymt_col, chrg_col, svc_col)

if pymt_col:  df_out_w['avg_paid']   = parse_money(df_out_w[pymt_col])
if chrg_col:  df_out_w['avg_billed'] = parse_money(df_out_w[chrg_col])
if svc_col:   df_out_w['svc_count']  = pd.to_numeric(df_out_w[svc_col], errors='coerce')

if 'avg_paid' in df_out_w.columns and 'avg_billed' in df_out_w.columns:
    df_out_w['yield_out'] = df_out_w['avg_paid'] / df_out_w['avg_billed'].replace(0, np.nan)
    df_out_w['variance_out'] = df_out_w['avg_billed'] - df_out_w['avg_paid']

con.register('out_work', df_out_w)

# APC summary
if 'APC_Cd' in df_out_w.columns and 'avg_paid' in df_out_w.columns:
    df_apc_summ = con.execute("""
        SELECT APC_Cd, COUNT(*) AS provider_count,
               ROUND(AVG(avg_paid),2) AS avg_paid,
               ROUND(AVG(yield_out)*100,2) AS avg_yield_pct
        FROM out_work WHERE avg_paid > 0
        GROUP BY APC_Cd ORDER BY avg_paid DESC LIMIT 15
    """).df()
    print("\n🏥 OUTPATIENT APC SUMMARY (Top 15 by avg payment)")
    print(df_apc_summ.to_string(index=False))
else:
    print("Outpatient columns found:", list(df_out_w.columns[:20]))
    print("\nRunning PFS Part-B benchmark analysis instead...")
    if 'TOTAL_RVU_STD' in df_pfs.columns:
        print(df_pfs[['TOTAL_RVU_STD','MEDICARE_PMT']].describe().apply("${:,.2f}".format))


## 11. NY SPARCS Multi-Payer Analysis <a id='sparcs'></a>

In [ ]:
# ── Payer volume & cost summary ───────────────────────────────────────────────
df_payer_summ = con.execute("""
    SELECT PAYER_TIER,
           COUNT(*)                             AS discharges,
           ROUND(SUM(TOTAL_CHARGES),0)          AS total_charges,
           ROUND(SUM(TOTAL_COSTS),0)            AS total_costs,
           ROUND(AVG(TOTAL_CHARGES),0)          AS avg_charges,
           ROUND(AVG(TOTAL_COSTS),0)            AS avg_costs,
           ROUND(AVG(LOS),2)                    AS avg_los,
           ROUND(SUM(TOTAL_COSTS)/NULLIF(SUM(TOTAL_CHARGES),0)*100,2) AS ccr_pct
    FROM sparcs_raw
    WHERE PAYER_TIER IS NOT NULL AND TOTAL_CHARGES > 0
    GROUP BY PAYER_TIER ORDER BY discharges DESC
""").df()
print("💰 NY SPARCS 2022 — MULTI-PAYER SUMMARY")
print(df_payer_summ.to_string(index=False))


In [ ]:
# ── MDC × Payer analysis (NOTE: APR-DRG ≠ MS-DRG — join at MDC level only) ──
# IMPORTANT: Never join APR-DRG codes to MS-DRG codes numerically.
# APR-DRG and MS-DRG use independent numbering systems.
# Valid cross-walk is at MDC level (both share same 25 MDC codes).

if all(c in df_sparcs.columns for c in ['APR_MDC_CD','PAYER_TIER','TOTAL_CHARGES']):
    df_mdc_payer = con.execute("""
        SELECT APR_MDC_CD, MDC_DESC, PAYER_TIER,
               COUNT(*) AS discharges,
               ROUND(AVG(TOTAL_CHARGES),0) AS avg_charges,
               ROUND(AVG(TOTAL_COSTS),0)   AS avg_costs,
               ROUND(AVG(LOS),2)           AS avg_los,
               ROUND(AVG(CAST(SEVERITY AS INT)),2) AS avg_severity
        FROM sparcs_raw
        WHERE PAYER_TIER IN ('Medicare','Medicaid','Commercial_BCBS','Commercial_Private')
          AND APR_MDC_CD NOT IN ('','nan','None')
        GROUP BY APR_MDC_CD, MDC_DESC, PAYER_TIER
        ORDER BY APR_MDC_CD, PAYER_TIER
    """).df()
    # Pivot to compare payers side by side
    if len(df_mdc_payer) > 0:
        piv = df_mdc_payer.pivot_table(
            index=['APR_MDC_CD','MDC_DESC'], columns='PAYER_TIER',
            values='avg_charges', aggfunc='mean').round(0)
        print("\n📊 AVG CHARGES BY MDC × PAYER (pivot)")
        print(piv.head(12).to_string())


## 12. Contract Scenario Modeling <a id='scenarios'></a>

In [ ]:
# ── Contract scenario definitions ─────────────────────────────────────────────
SCENARIOS = {
    "baseline": {
        "label": "Baseline — Medicare FFS (100%)",
        "pct_medicare": 1.00, "stop_loss_thresh": 100_000,
        "sl_pct": 0.60, "denial_rate": 0.00,
    },
    "commercial_190": {
        "label": "Commercial Current — 190% Medicare",
        "pct_medicare": 1.90, "stop_loss_thresh": 100_000,
        "sl_pct": 0.60, "denial_rate": 0.08,
    },
    "s1_rate_escalation": {
        "label": "Scenario 1 — Rate Escalation 215% Medicare",
        "pct_medicare": 2.15, "stop_loss_thresh": 100_000,
        "sl_pct": 0.60, "denial_rate": 0.08,
    },
    "s2_stop_loss": {
        "label": "Scenario 2 — Stop-Loss Reduced ($75K)",
        "pct_medicare": 1.90, "stop_loss_thresh": 75_000,
        "sl_pct": 0.65, "denial_rate": 0.08,
    },
    "s3_denial_cut": {
        "label": "Scenario 3 — Denial Rate Halved (4%)",
        "pct_medicare": 1.90, "stop_loss_thresh": 100_000,
        "sl_pct": 0.60, "denial_rate": 0.04,
    },
    "s4_carve_out": {
        "label": "Scenario 4 — High-Weight DRG Carve-Out 125%",
        "pct_medicare": 1.90, "stop_loss_thresh": 100_000,
        "sl_pct": 0.60, "denial_rate": 0.08,
        "carve_thresh": 3.0, "carve_pct": 1.25,
    },
}

def run_scenario(df: pd.DataFrame, s: dict) -> dict:
    d = df.dropna(subset=['expected_payment','tot_dschrgs']).copy()
    base = s['pct_medicare']
    if 'carve_thresh' in s:
        d['rate'] = np.where(d['drg_weight']>s['carve_thresh'],
                             d['expected_payment']*s['carve_pct'],
                             d['expected_payment']*base)
    else:
        d['rate'] = d['expected_payment'] * base
    sl_add = np.where(d['avg_billed']>s['stop_loss_thresh'],
                      (d['avg_billed']-s['stop_loss_thresh'])*s['sl_pct'], 0.0)
    d['net'] = (d['rate'] + sl_add) * (1 - s['denial_rate'])
    total_rev = (d['net'] * d['tot_dschrgs']).sum()
    total_med = (d['expected_payment'] * d['tot_dschrgs']).sum()
    total_act = (d['avg_medicare_pymt'] * d['tot_dschrgs']).sum()
    vols      = d['tot_dschrgs'].sum()
    cmi       = (d['drg_weight']*d['tot_dschrgs']).sum()/vols
    return {"label":s['label'], "total_revenue":total_rev,
            "vs_medicare":total_rev-total_med, "vs_actual":total_rev-total_act,
            "cmi":round(cmi,4), "avg_per_disch":total_rev/vols if vols>0 else 0}

results = {}
for k, s in SCENARIOS.items():
    results[k] = run_scenario(df_variance, s)
    print(f"  ✅ {s['label']}")

df_scen = pd.DataFrame(results).T
print("\n🎯 SCENARIO RESULTS")
for k, v in results.items():
    print(f"  {v['label'][:45]:<45} | Rev: ${v['total_revenue']:>16,.0f} | Delta vs Actual: ${v['vs_actual']:>14,.0f} | CMI: {v['cmi']:.4f}")


In [ ]:
# ── CMI impact analysis ───────────────────────────────────────────────────────
# Every 0.10 CMI increase on 10,000 Medicare discharges ≈ $6.2M additional revenue
total_discharges = df_variance['tot_dschrgs'].sum()
base_cmi = (df_variance['drg_weight']*df_variance['tot_dschrgs']).sum()/total_discharges
cmi_delta = 0.10
revenue_per_cmi_point = cmi_delta * IPPS['BASE_RATE'] * total_discharges

print(f"\n📊 CASE-MIX INDEX ANALYSIS")
print(f"  Current national CMI:        {base_cmi:.4f}")
print(f"  Total discharges analyzed:   {total_discharges:,.0f}")
print(f"  Revenue impact of +0.10 CMI: ${revenue_per_cmi_point:,.0f}")
print(f"  (National benchmark: ~1.79 for Medicare hospitals — Pi et al., PMC10785921)")


## 13. Visualizations <a id='viz'></a>

In [ ]:
# ── Chart 1: Top 15 DRGs — horizontal bar ────────────────────────────────────
if len(df_top_drgs) > 0:
    fig1 = px.bar(
        df_top_drgs.head(15),
        x='total_underpayment', y='drg_desc', orientation='h',
        color='avg_yield_pct', color_continuous_scale='RdBu',
        color_continuous_midpoint=96,
        title='🔴 Top 15 DRGs by Total Underpayment Exposure (FY2022)',
        labels={'total_underpayment':'Total Underpayment ($)','drg_desc':'DRG','avg_yield_pct':'Yield %'},
        text='total_underpayment',
    )
    fig1.update_traces(texttemplate='$%{x:,.0f}', textposition='outside')
    fig1.update_layout(height=560, yaxis={'categoryorder':'total ascending'},
                       plot_bgcolor='white', coloraxis_colorbar_title='Yield %')
    fig1.show()
    fig1.write_html(str(OUTPUT_DIR / "chart1_top_drgs.html"))
    print("✅ Chart 1 saved")


In [ ]:
# ── Chart 2: Yield by MDC service line ───────────────────────────────────────
df_mdc_y = con.execute("""
    SELECT mdc, ROUND(AVG(yield_ratio)*100,2) AS avg_yield,
           ROUND(STDDEV(yield_ratio)*100,2) AS std_yield,
           SUM(underpayment_flag) AS underpaid_count
    FROM variance WHERE mdc NOT IN ('00','XX','nan','None','')
    GROUP BY mdc ORDER BY avg_yield ASC
""").df()

if len(df_mdc_y) > 0:
    fig2 = go.Figure()
    colors = ['#d62728' if v < 92 else '#1f77b4' for v in df_mdc_y['avg_yield']]
    fig2.add_trace(go.Bar(x=df_mdc_y['mdc'], y=df_mdc_y['avg_yield'],
                          error_y=dict(type='data', array=df_mdc_y['std_yield']),
                          marker_color=colors, name='Avg Yield %'))
    fig2.add_hline(y=92, line_dash="dash", line_color="red",  annotation_text="92% threshold")
    fig2.add_hline(y=96, line_dash="dot",  line_color="orange", annotation_text="96% monitor")
    fig2.update_layout(title='📊 Yield by MDC (Red = Below 92%)',
                       xaxis_title='MDC Code', yaxis_title='Avg Yield (%)',
                       plot_bgcolor='white', height=420)
    fig2.show()
    fig2.write_html(str(OUTPUT_DIR / "chart2_yield_mdc.html"))
    print("✅ Chart 2 saved")


In [ ]:
# ── Chart 3: US State underpayment map ───────────────────────────────────────
if len(df_state_rank) > 0:
    fig3 = px.choropleth(
        df_state_rank, locations='state', locationmode='USA-states',
        color='total_underpayment', scope='usa',
        color_continuous_scale='Reds',
        title='🗺️ Total Underpayment by State (FY2022)',
        labels={'total_underpayment':'Underpayment ($)'})
    fig3.update_layout(height=480)
    fig3.show()
    fig3.write_html(str(OUTPUT_DIR / "chart3_state_map.html"))
    print("✅ Chart 3 saved")


In [ ]:
# ── Chart 4: Scenario tornado chart ─────────────────────────────────────────
s_labels = [v['label'] for v in results.values()]
s_deltas = [v['vs_actual'] for v in results.values()]

fig4 = go.Figure(go.Bar(
    x=s_deltas, y=s_labels, orientation='h',
    marker_color=['#2166ac' if d>=0 else '#d62728' for d in s_deltas],
    text=[f"${d:,.0f}" for d in s_deltas], textposition='outside'))
fig4.add_vline(x=0, line_width=2, line_color='black')
fig4.update_layout(
    title='🎯 Contract Renegotiation Scenario Impact vs. Current Actuals',
    xaxis_title='Revenue Delta vs. Actual ($)', plot_bgcolor='white', height=380)
fig4.show()
fig4.write_html(str(OUTPUT_DIR / "chart4_scenarios.html"))
print("✅ Chart 4 saved")


In [ ]:
# ── Chart 5: Expected vs Actual scatter (sample 5K) ──────────────────────────
df_sc = df_variance.dropna(subset=['expected_payment','avg_medicare_pymt','drg_weight'])
df_sc = df_sc.sample(min(5000,len(df_sc)), random_state=42)

fig5 = px.scatter(
    df_sc, x='expected_payment', y='avg_medicare_pymt',
    color='yield_ratio', color_continuous_scale='RdBu', color_continuous_midpoint=1.0,
    opacity=0.45, size='drg_weight', size_max=9,
    title='📈 Expected vs. Actual Medicare Payment (colour=yield, size=DRG weight)',
    labels={'expected_payment':'Expected IPPS ($)','avg_medicare_pymt':'Actual Medicare ($)','yield_ratio':'Yield'})
maxv = max(df_sc['expected_payment'].max(), df_sc['avg_medicare_pymt'].max())
fig5.add_trace(go.Scatter(x=[0,maxv], y=[0,maxv], mode='lines',
               line=dict(color='black',dash='dash'), name='Expected=Actual'))
fig5.update_layout(height=480, plot_bgcolor='white')
fig5.show()
fig5.write_html(str(OUTPUT_DIR / "chart5_scatter.html"))
print("✅ Chart 5 saved")

# ── Chart 6: NY SPARCS payer mix ─────────────────────────────────────────────
if len(df_payer_summ) > 0:
    df_pm = df_payer_summ[df_payer_summ['discharges']>500].copy()
    fig6 = go.Figure()
    fig6.add_trace(go.Bar(name='Avg Charges', x=df_pm['PAYER_TIER'],
                          y=df_pm['avg_charges'], marker_color='#4C72B0'))
    fig6.add_trace(go.Bar(name='Avg Costs', x=df_pm['PAYER_TIER'],
                          y=df_pm['avg_costs'], marker_color='#DD8452'))
    fig6.update_layout(barmode='group', plot_bgcolor='white', height=420,
                       title='💰 NY SPARCS 2022 — Avg Charges vs. Costs by Payer Tier')
    fig6.show()
    fig6.write_html(str(OUTPUT_DIR / "chart6_sparcs_payer.html"))
    print("✅ Chart 6 saved")


## 14. Export (Excel, Parquet, MySQL) <a id='export'></a>

In [ ]:
# ── Build analytical marts ────────────────────────────────────────────────────
mart_variance = df_variance[[
    'ccn','drg_cd','drg_desc','mdc','drg_type','state','city',
    'drg_weight','gmlos','wage_index','ime_adj','dsh_adj',
    'tot_dschrgs','avg_billed','avg_total_pymt','avg_medicare_pymt',
    'expected_payment','expected_total','actual_total',
    'variance_dollars','variance_pct','yield_ratio',
    'underpayment_flag','overpayment_flag','stop_loss_flag',
    'high_priority','lesser_of_flag',
]].copy()

mart_scenarios = pd.DataFrame([
    {'scenario':k, 'label':v['label'], 'total_revenue':v['total_revenue'],
     'vs_medicare_uplift':v['vs_medicare'], 'vs_actual_uplift':v['vs_actual'],
     'cmi':v['cmi'], 'avg_per_discharge':v['avg_per_disch']}
    for k,v in results.items()])

MARTS = {
    "fct_payment_variance":   mart_variance,
    "dim_state_underpayment": df_state_rank,
    "dim_drg_yield_bench":    df_yield,
    "mart_scenario_results":  mart_scenarios,
    "mart_cmi_by_state":      df_cmi,
    "mart_sparcs_payer":      df_payer_summ,
}

print("✅ Analytical marts:")
for n, d in MARTS.items():
    print(f"   {n}: {len(d):,} rows × {d.shape[1]} cols")


In [ ]:
# ── Export to Excel ───────────────────────────────────────────────────────────
xlsx_path = OUTPUT_DIR / "payment_variance_analytics.xlsx"
with pd.ExcelWriter(xlsx_path, engine='openpyxl') as w:
    for name, df in MARTS.items():
        de = df.drop(columns=['_source','_dq_flags'], errors='ignore')
        de.to_excel(w, sheet_name=name[:31], index=False)
        print(f"  ✅ Sheet '{name[:31]}': {len(de):,} rows")
print(f"\n📊 Excel saved: {xlsx_path}")


In [ ]:
# ── Export to Parquet ─────────────────────────────────────────────────────────
for name, df in MARTS.items():
    p = OUTPUT_DIR / f"{name}.parquet"
    de = df.drop(columns=['_source','_dq_flags'], errors='ignore').copy()
    for col in de.select_dtypes(include='object').columns:
        de[col] = de[col].astype(str)
    de.to_parquet(p, engine='pyarrow', compression='snappy', index=False)
print(f"✅ Parquet files saved to {OUTPUT_DIR}")

# ── Push to MySQL ─────────────────────────────────────────────────────────────
if MYSQL_OK:
    for name, df in MARTS.items():
        push_mysql(df, name, engine)
    # Create underpayment priority view
    with engine.connect() as conn:
        conn.execute(text("""
            CREATE OR REPLACE VIEW v_underpayment_priority AS
            SELECT ccn, drg_cd, drg_desc, state, mdc,
                   variance_dollars, yield_ratio, stop_loss_flag, high_priority
            FROM fct_payment_variance
            WHERE underpayment_flag=1 AND high_priority=1
            ORDER BY variance_dollars DESC
        """))
        conn.commit()
    print("✅ MySQL: all marts pushed + priority view created")
else:
    print("ℹ️  MySQL not connected — marts saved as Excel + Parquet only")


## 15. Executive Summary <a id='summary'></a>

In [ ]:
from IPython.display import Markdown, display

# Compute headline numbers
tot_und  = df_variance.loc[df_variance['underpayment_flag']==1,'variance_dollars'].sum()
tot_ove  = df_variance.loc[df_variance['overpayment_flag']==1,'variance_dollars'].abs().sum()
n_und    = int(df_variance['underpayment_flag'].sum())
n_hi     = int(df_variance['high_priority'].sum())
n_sl     = int(df_variance['stop_loss_flag'].sum())
nat_cmi  = (df_variance['drg_weight']*df_variance['tot_dschrgs']).sum()/df_variance['tot_dschrgs'].sum()
tot_disc = int(df_variance['tot_dschrgs'].sum())
best_s   = max(results, key=lambda k: results[k]['vs_actual'])

scen_rows = "\n".join(f"| {v['label']:<50} | ${v['vs_actual']:>16,.0f} |"
                       for v in results.values())

md_text = f"""
---
# 📋 EXECUTIVE SUMMARY — Healthcare Payment Variance Analytics
**Peter Chika Ozo-ogueji | American University | Guidehouse Interview | May 1, 2026**

---
## Key Metrics (CMS Medicare FY2022)

| Metric | Value |
|---|---|
| Total Medicare Discharges Analyzed | {tot_disc:,} |
| National Case-Mix Index | {nat_cmi:.4f} |
| Hospital×DRG Underpayment Flags | {n_und:,} |
| **Total Underpayment Exposure** | **${tot_und:,.0f}** |
| Total Overpayment Detected | ${tot_ove:,.0f} |
| High-Priority Accounts (Yield < 92%) | {n_hi:,} |
| Stop-Loss Outlier Accounts | {n_sl:,} |

---
## Contract Scenario Results

| Scenario | Revenue Delta vs. Actual |
|---|---|
{scen_rows}

**Best opportunity:** {results[best_s]['label']} → ${results[best_s]['vs_actual']:,.0f} uplift

---
## Methodology
| Item | Detail |
|---|---|
| FY2025 IPPS Base Rate | **$6,624.39** (IFC-corrected Oct 2024) |
| IME + DSH Application | **Additive** `(1 + IME + DSH)` |
| CCN/DRG Keys | Zero-padded strings — never cast to int |
| Variance Threshold | \|Var%\| > 5% AND \|Var$\| > $200 (dual) |
| False-Variance Logic | Lesser-of, stop-loss, transfer DRG applied |
| SQL Engine | DuckDB (window functions, CTEs) |
| Persistence | MySQL + Parquet on Google Drive |

*Outputs: {OUTPUT_DIR}*
"""
display(Markdown(md_text))
